## Name: Asit Jain
## Roll No: M25DE1049
## Assignment 3 - MLBD

# Part 6: Explainability in Recommender Systems
## Task 10: Feature-Based Explanations (For Content-Based Filtering)

Show which features contributed the most to a recommendation using SHAP (SHapley Additive Explanations).

Example: *"This movie was recommended because you liked Sci-Fi movies from the 2010s."*

**Approach:** Train a content-based model, then use SHAP to explain feature contributions for each prediction.

**Dataset:** MovieLens ml-latest-small

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'scikit-learn', 'shap', 'matplotlib', '-q'])

import os, zipfile, urllib.request, re, warnings
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')
print(f'SHAP {shap.__version__} - All imports successful!')

### Step 1: Load and Prepare Data

In [ ]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'
zip_path = 'ml-latest-small.zip'
extract_dir = 'ml-latest-small'
if not os.path.exists(extract_dir):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z: z.extractall('.')

movies = pd.read_csv(os.path.join(extract_dir, 'movies.csv'))
ratings = pd.read_csv(os.path.join(extract_dir, 'ratings.csv'))
print(f'Movies: {len(movies)}, Ratings: {len(ratings)}')

### Step 2: Build Content Features

For each (user, movie) pair, build a feature vector combining:
- Movie genre indicators
- Movie release year
- Movie average rating
- User's average rating per genre (preference profile)

In [ ]:
# Extract year
def extract_year(title):
    match = re.search(r'\((\d{4})\)', str(title))
    return int(match.group(1)) if match else np.nan

movies['year'] = movies['title'].apply(extract_year)
movies['year'] = movies['year'].fillna(movies['year'].median())

# One-hot genres
all_genres = sorted(set(g for gs in movies['genres'] for g in gs.split('|') if g != '(no genres listed)'))
for genre in all_genres:
    movies[f'g_{genre}'] = movies['genres'].apply(lambda x, g=genre: 1.0 if g in x.split('|') else 0.0)
genre_cols = [f'g_{g}' for g in all_genres]

# Movie avg rating
movie_avg = ratings.groupby('movieId')['rating'].mean()
movies['avg_rating'] = movies['movieId'].map(movie_avg).fillna(ratings['rating'].mean())

# User genre preferences (avg rating per genre)
ratings_g = ratings.merge(movies[['movieId'] + genre_cols], on='movieId')
user_genre_prefs = {}
for uid, group in ratings_g.groupby('userId'):
    prefs = {}
    for g in all_genres:
        mask = group[f'g_{g}'] == 1.0
        prefs[f'user_pref_{g}'] = group.loc[mask, 'rating'].mean() if mask.sum() > 0 else 0.0
    user_genre_prefs[uid] = prefs
user_prefs_df = pd.DataFrame.from_dict(user_genre_prefs, orient='index')
user_pref_cols = list(user_prefs_df.columns)
print(f'Genres: {len(all_genres)}, User pref features: {len(user_pref_cols)}')

### Step 3: Build Combined Feature Matrix for (User, Movie) Pairs

In [ ]:
# Build feature matrix: movie features + user preference features
movie_feat = movies.set_index('movieId')[genre_cols + ['year', 'avg_rating']]
feature_rows = []
labels = []

for _, row in ratings.iterrows():
    uid, mid, rating = int(row['userId']), int(row['movieId']), row['rating']
    if mid not in movie_feat.index or uid not in user_prefs_df.index:
        continue
    mf = movie_feat.loc[mid].to_dict()
    uf = user_prefs_df.loc[uid].to_dict()
    feature_rows.append({**mf, **uf})
    labels.append(rating)

X = pd.DataFrame(feature_rows)
y = np.array(labels)
print(f'Feature matrix: {X.shape}, Labels: {y.shape}')
print(f'Features: {list(X.columns[:5])} ... {list(X.columns[-5:])}')

### Step 4: Train a Content-Based Model (Gradient Boosting)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f'Content-Based GBR RMSE: {rmse:.4f}')

### Step 5: SHAP Global Feature Importance

Which features matter most across all predictions?

In [ ]:
# Compute SHAP values using TreeExplainer (fast for tree models)
explainer = shap.TreeExplainer(model)
# Use a sample for speed
X_sample = X_test.sample(500, random_state=42)
shap_values = explainer.shap_values(X_sample)

# Global feature importance (mean |SHAP|)
shap.summary_plot(shap_values, X_sample, plot_type='bar', max_display=15, show=False)
plt.title('SHAP Global Feature Importance (Content-Based Model)')
plt.tight_layout()
plt.show()

### Step 6: SHAP Beeswarm Plot (Feature Impact Direction)

In [ ]:
# Beeswarm: shows direction of feature impact
shap.summary_plot(shap_values, X_sample, max_display=15, show=False)
plt.title('SHAP Feature Impact on Predicted Rating')
plt.tight_layout()
plt.show()

### Step 7: SHAP Explanations for Individual Recommendations

Generate human-readable explanations for specific (user, movie) predictions.

In [ ]:
def explain_recommendation(idx, X_data, shap_vals, top_n=5):
    """Generate a human-readable explanation for a single prediction."""
    row = X_data.iloc[idx]
    sv = shap_vals[idx]
    pred = model.predict(row.values.reshape(1, -1))[0]

    # Sort features by absolute SHAP value
    feat_impact = sorted(zip(X_data.columns, sv, row.values), key=lambda x: abs(x[1]), reverse=True)

    print(f'Predicted Rating: {pred:.2f}')
    base_val = explainer.expected_value
    if hasattr(base_val, '__len__'):
        base_val = base_val[0]
    print(f'Base value: {float(base_val):.2f}')
    print(f'\nTop {top_n} contributing features:')
    for feat, shap_v, feat_v in feat_impact[:top_n]:
        direction = '\u2191' if shap_v > 0 else '\u2193'
        # Make feature name readable
        name = feat.replace('g_', 'Genre: ').replace('user_pref_', 'Your preference for ')
        print(f'  {direction} {name} = {feat_v:.2f} (impact: {shap_v:+.3f})')

    # Generate natural language explanation
    top_pos = [(f, s, v) for f, s, v in feat_impact if s > 0][:3]
    reasons = []
    for feat, shap_v, feat_v in top_pos:
        if feat.startswith('g_'):
            reasons.append(f'it is a {feat[2:]} movie')
        elif feat.startswith('user_pref_'):
            reasons.append(f'you enjoy {feat[10:]} films (avg {feat_v:.1f} stars)')
        elif feat == 'avg_rating':
            reasons.append(f'it is highly rated ({feat_v:.1f} stars avg)')
        elif feat == 'year':
            reasons.append(f'it was released in {int(feat_v)}')
    if reasons:
        print(f'\nThis movie was recommended because {" and ".join(reasons)}.')

# Explain 3 sample predictions
for i in [0, 10, 50]:
    print('='*65)
    print(f'Explanation for sample #{i}')
    print('='*65)
    explain_recommendation(i, X_sample, shap_values)
    print()

### Step 8: SHAP Waterfall Plot for a Single Prediction

In [ ]:
# Waterfall plot: shows how each feature pushes prediction from base value
base_v = explainer.expected_value
if hasattr(base_v, '__len__'):
    base_v = float(base_v[0])
else:
    base_v = float(base_v)

shap_explanation = shap.Explanation(
    values=shap_values[0],
    base_values=base_v,
    data=X_sample.iloc[0].values,
    feature_names=X_sample.columns.tolist())

shap.waterfall_plot(shap_explanation, max_display=12, show=False)
plt.title('SHAP Waterfall: Single Recommendation Breakdown')
plt.tight_layout()
plt.show()

### Step 9: Feature Importance by Category

In [ ]:
# Group SHAP values by feature category
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feat_importance = pd.Series(mean_abs_shap, index=X_sample.columns)

categories = {'Movie Genres': [], 'User Preferences': [], 'Movie Metadata': []}
for feat in X_sample.columns:
    if feat.startswith('g_'):
        categories['Movie Genres'].append(feat)
    elif feat.startswith('user_pref_'):
        categories['User Preferences'].append(feat)
    else:
        categories['Movie Metadata'].append(feat)

cat_importance = {cat: feat_importance[feats].sum() for cat, feats in categories.items()}

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2196F3', '#FF9800', '#4CAF50']
bars = ax.barh(list(cat_importance.keys()), list(cat_importance.values()), color=colors, edgecolor='black')
ax.set_xlabel('Total Mean |SHAP Value|')
ax.set_title('Feature Importance by Category')
for bar, val in zip(bars, cat_importance.values()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')
plt.tight_layout()
plt.show()

print('\nCategory breakdown:')
for cat, val in sorted(cat_importance.items(), key=lambda x: -x[1]):
    pct = val / sum(cat_importance.values()) * 100
    print(f'  {cat}: {val:.3f} ({pct:.1f}%)')

### Analysis

**Key Findings:**

1. **SHAP reveals feature contributions**: Unlike black-box predictions, SHAP shows exactly which features pushed a rating prediction up or down. User genre preferences and movie genre indicators are typically the strongest drivers.

2. **Human-readable explanations**: By mapping SHAP values to natural language, we make recommendations transparent and trustworthy.

3. **Category-level insights**: Grouping features shows whether the model relies more on user preferences, movie genres, or metadata like year/popularity. This helps identify potential biases.

4. **Per-prediction transparency**: Waterfall plots show the exact path from the base prediction to the final score, making each recommendation auditable.